In [1]:
# ============================================================
# NOTEBOOK 3: EMBEDDINGS + VECTOR DATABASE BUILD
# Project: AgriGenius
#
# Purpose:
# Build a ChromaDB vector store from chunked agricultural text.
#
# Input:
#   outputs/run_<RUN_ID>/chunks/chunks.parquet
#
# Output:
#   outputs/run_<RUN_ID>/vectordb/chroma_db/
#   outputs/run_<RUN_ID>/vectordb/vector_store_report.csv
#   outputs/run_<RUN_ID>/vectordb/vector_store_log.json
#
# Notes:
# - ChromaDB is first built locally in /content to avoid Google Drive I/O issues.
# - After indexing, the database is copied to Drive.
# ============================================================


# -----------------------------
# Cell 1 — Install dependencies
# -----------------------------
!pip -q install chromadb==0.5.5 sentence-transformers==3.0.1 pandas pyarrow


# -----------------------------
# Cell 2 — Imports
# -----------------------------
import os
import json
import shutil
import time
from datetime import datetime

import pandas as pd
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings


# -----------------------------
# Cell 3 — Mount Google Drive
# -----------------------------
from google.colab import drive
drive.mount("/content/drive", force_remount=True)



/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


Mounted at /content/drive


In [2]:

# -----------------------------
# Cell 4 — Project paths
# -----------------------------
PROJECT_ROOT = "/content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project"

# ✅ Use the same RUN_ID generated in earlier notebooks
RUN_ID = "20260209_185402"   # change only if needed

RUN_DIR = os.path.join(PROJECT_ROOT, "outputs", f"run_{RUN_ID}")
CHUNKS_DIR = os.path.join(RUN_DIR, "chunks")
CHUNKS_PARQUET = os.path.join(CHUNKS_DIR, "chunks.parquet")

VECTOR_DIR = os.path.join(RUN_DIR, "vectordb")
os.makedirs(VECTOR_DIR, exist_ok=True)

# Build DB locally first (safe for Colab)
CHROMA_DIR_LOCAL = f"/content/chroma_db_{RUN_ID}"

# Final DB copy stored in Drive
CHROMA_DIR_DRIVE = os.path.join(VECTOR_DIR, "chroma_db")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUN_DIR:", RUN_DIR)
print("CHUNKS_PARQUET:", CHUNKS_PARQUET, "| exists:", os.path.exists(CHUNKS_PARQUET))
print("CHROMA_DIR_LOCAL:", CHROMA_DIR_LOCAL)
print("CHROMA_DIR_DRIVE:", CHROMA_DIR_DRIVE)


# -----------------------------
# Cell 5 — Load chunks.parquet
# -----------------------------
if not os.path.exists(CHUNKS_PARQUET):
    raise FileNotFoundError(f"Missing input file: {CHUNKS_PARQUET}\nRun Notebook 2 first.")

chunks_df = pd.read_parquet(CHUNKS_PARQUET)

required_cols = [
    "chunk_id",
    "chunk_text",
    "source_type",
    "source_name",
    "file_name",
    "chunk_index_in_file",
    "chunk_words",
    "chunk_chars"
]
missing_cols = [c for c in required_cols if c not in chunks_df.columns]
if missing_cols:
    raise ValueError(f"chunks.parquet is missing required columns: {missing_cols}")

# Clean blanks
chunks_df["chunk_text"] = chunks_df["chunk_text"].fillna("").astype(str)
chunks_df = chunks_df[chunks_df["chunk_text"].str.strip() != ""].copy()

if len(chunks_df) == 0:
    raise ValueError("No valid chunks found after cleaning.")

print(f"✅ Loaded {len(chunks_df)} chunks")
display(chunks_df.head(5))


# -----------------------------
# Cell 6 — Load embedding model
# -----------------------------
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

print("Loading embedding model...")
t0_model = time.time()
embedder = SentenceTransformer(EMBED_MODEL_NAME)
model_load_time_sec = round(time.time() - t0_model, 2)

print("✅ Embedding model loaded:", EMBED_MODEL_NAME)
print("Model load time (sec):", model_load_time_sec)


# -----------------------------
# Cell 7 — Create fresh local Chroma DB
# -----------------------------
# Remove old local DB if present
if os.path.exists(CHROMA_DIR_LOCAL):
    shutil.rmtree(CHROMA_DIR_LOCAL)

client = chromadb.PersistentClient(
    path=CHROMA_DIR_LOCAL,
    settings=Settings(anonymized_telemetry=False)
)

COLLECTION_NAME = f"agrigenius_{RUN_ID}"

# Ensure fresh collection
try:
    client.delete_collection(COLLECTION_NAME)
except Exception:
    pass

collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={
        "project": "AgriGenius",
        "run_id": RUN_ID,
        "embedding_model": EMBED_MODEL_NAME,
        "hnsw:space": "cosine"
    }
)

print("✅ Chroma collection ready:", COLLECTION_NAME)


# -----------------------------
# Cell 8 — Prepare documents and metadata
# -----------------------------
docs = chunks_df["chunk_text"].astype(str).tolist()
ids = chunks_df["chunk_id"].astype(str).tolist()

metadatas = []
for _, row in chunks_df.iterrows():
    metadatas.append({
        "source_type": str(row["source_type"]),
        "source_name": str(row["source_name"]),
        "file_name": str(row["file_name"]),
        "chunk_index_in_file": int(row["chunk_index_in_file"]),
        "chunk_words": int(row["chunk_words"]),
        "chunk_chars": int(row["chunk_chars"])
    })

print("✅ Prepared for vector indexing")
print("Documents:", len(docs))
print("IDs:", len(ids))
print("Metadata records:", len(metadatas))
print("Example metadata:", metadatas[0])


# -----------------------------
# Cell 9 — Embed and index in batches
# -----------------------------
BATCH_SIZE = 96

def batch_ranges(total_size, batch_size):
    for start in range(0, total_size, batch_size):
        end = min(start + batch_size, total_size)
        yield start, end

print("\n" + "=" * 90)
print("STEP: EMBEDDING + INDEXING INTO CHROMA DB")
print("=" * 90)

t0_index = time.time()

for start, end in batch_ranges(len(docs), BATCH_SIZE):
    batch_docs = docs[start:end]
    batch_ids = ids[start:end]
    batch_meta = metadatas[start:end]

    batch_embeddings = embedder.encode(
        batch_docs,
        show_progress_bar=False
    ).tolist()

    collection.add(
        ids=batch_ids,
        documents=batch_docs,
        metadatas=batch_meta,
        embeddings=batch_embeddings
    )

    print(f"Indexed {end:>5}/{len(docs)}")

index_time_sec = round(time.time() - t0_index, 2)

print("\n✅ Vector DB indexing complete.")
print("Total vectors in collection:", collection.count())
print("Indexing time (sec):", index_time_sec)


# -----------------------------
# Cell 10 — Copy local DB to Drive
# -----------------------------
# Remove old DB in Drive if it exists
if os.path.exists(CHROMA_DIR_DRIVE):
    shutil.rmtree(CHROMA_DIR_DRIVE)

shutil.copytree(CHROMA_DIR_LOCAL, CHROMA_DIR_DRIVE)

print("✅ Chroma DB copied to Drive successfully")
print("LOCAL DB:", CHROMA_DIR_LOCAL)
print("DRIVE DB:", CHROMA_DIR_DRIVE)


# -----------------------------
# Cell 11 — Create vector store report
# -----------------------------
report_df = chunks_df.groupby("source_type", as_index=False).agg(
    files=("file_name", "nunique"),
    chunks=("chunk_id", "count"),
    avg_chunk_words=("chunk_words", "mean"),
    avg_chunk_chars=("chunk_chars", "mean")
)

report_df["avg_chunk_words"] = report_df["avg_chunk_words"].round(2)
report_df["avg_chunk_chars"] = report_df["avg_chunk_chars"].round(2)

report_df = report_df.sort_values("chunks", ascending=False).reset_index(drop=True)

report_path = os.path.join(VECTOR_DIR, "vector_store_report.csv")
report_df.to_csv(report_path, index=False)

print("✅ Vector store report saved:", report_path)
display(report_df)


# -----------------------------
# Cell 12 — Save reproducibility log
# -----------------------------
embedding_dim = len(embedder.encode(["test"], show_progress_bar=False)[0])

log_data = {
    "run_id": RUN_ID,
    "created_at": datetime.now().isoformat(),
    "project": "AgriGenius",
    "input_chunks_file": CHUNKS_PARQUET,
    "chunks_indexed": int(len(chunks_df)),
    "collection_name": COLLECTION_NAME,
    "embedding_model": EMBED_MODEL_NAME,
    "embedding_dimension": int(embedding_dim),
    "vector_db": "ChromaDB",
    "similarity_space": "cosine",
    "batch_size": int(BATCH_SIZE),
    "model_load_time_sec": model_load_time_sec,
    "index_time_sec": index_time_sec,
    "local_persist_path": CHROMA_DIR_LOCAL,
    "drive_persist_path": CHROMA_DIR_DRIVE
}

log_path = os.path.join(VECTOR_DIR, "vector_store_log.json")
with open(log_path, "w", encoding="utf-8") as f:
    json.dump(log_data, f, indent=2)

print("✅ Vector store log saved:", log_path)
print(json.dumps(log_data, indent=2))


# -----------------------------
# Cell 13 — Quick verification search
# -----------------------------
def quick_search(query: str, k: int = 5):
    query_embedding = embedder.encode([query], show_progress_bar=False).tolist()

    result = collection.query(
        query_embeddings=query_embedding,
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )

    rows = []
    for i in range(len(result["ids"][0])):
        rows.append({
            "rank": i + 1,
            "chunk_id": result["ids"][0][i],
            "distance": float(result["distances"][0][i]),
            "source_type": result["metadatas"][0][i].get("source_type"),
            "file_name": result["metadatas"][0][i].get("file_name"),
            "source_name": result["metadatas"][0][i].get("source_name"),
            "preview": result["documents"][0][i][:180].replace("\n", " ") + "..."
        })
    return pd.DataFrame(rows)

print("\n" + "=" * 90)
print("DEMO: QUICK VECTOR SEARCH")
print("=" * 90)

demo_query = "What is Minimum Support Price scheme?"
demo_df = quick_search(demo_query, k=5)
display(demo_df)

PROJECT_ROOT: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project
RUN_DIR: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402
CHUNKS_PARQUET: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/chunks/chunks.parquet | exists: True
CHROMA_DIR_LOCAL: /content/chroma_db_20260209_185402
CHROMA_DIR_DRIVE: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/vectordb/chroma_db
✅ Loaded 891 chunks


,chunk_id,source_type,source_name,file_name,file_path,chunk_index_in_file,chunk_size_words,overlap_words,chunk_words,chunk_chars,chunk_preview,chunk_text
0,pdf_Farming_Schemes_0001_000001,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,1,220,50,220,1514,COMPENDIUM OF GOVERNMENT OF INDIA SCHEMES AND ...,COMPENDIUM OF GOVERNMENT OF INDIA SCHEMES AND ...
1,pdf_Farming_Schemes_0002_000002,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,2,220,50,220,1440,Gramin Agricultural Markets (GrAMs) 39 2. Mini...,Gramin Agricultural Markets (GrAMs) 39 2. Mini...
2,pdf_Farming_Schemes_0003_000003,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,3,220,50,220,1466,North East Special Infrastructure Development ...,North East Special Infrastructure Development ...
3,pdf_Farming_Schemes_0004_000004,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,4,220,50,220,1531,Operational Guidelines of the Scheme for Integ...,Operational Guidelines of the Scheme for Integ...
4,pdf_Farming_Schemes_0005_000005,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,5,220,50,220,1528,National TB Control Programme (RNTCP) 158 12. ...,National TB Control Programme (RNTCP) 158 12. ...


Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

...

Indexed    96/891
Indexed   192/891
Indexed   288/891
Indexed   384/891
Indexed   480/891
Indexed   576/891
Indexed   672/891
Indexed   768/891
Indexed   864/891
Indexed   891/891

✅ Vector DB indexing complete.
Total vectors in collection: 891
Indexing time (sec): 143.79
✅ Chroma DB copied to Drive successfully
LOCAL DB: /content/chroma_db_20260209_185402
DRIVE DB: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/vectordb/chroma_db
✅ Vector store report saved: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/vectordb/vector_store_report.csv


,source_type,files,chunks,avg_chunk_words,avg_chunk_chars
0,pdf,2,817,219.66,1427.60
1,web,10,74,207.27,1337.95


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


✅ Vector store log saved: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/vectordb/vector_store_log.json
{
  "run_id": "20260209_185402",
  "created_at": "2026-03-09T09:07:34.504193",
  "project": "AgriGenius",
  "input_chunks_file": "/content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/chunks/chunks.parquet",
  "chunks_indexed": 891,
  "collection_name": "agrigenius_20260209_185402",
  "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
  "embedding_dimension": 384,
  "vector_db": "ChromaDB",
  "similarity_space": "cosine",
  "batch_size": 96,
  "model_load_time_sec": 5.4,
  "index_time_sec": 143.79,
  "local_persist_path": "/content/chroma_db_20260209_185402",
  "drive_persist_path": "/content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/vectordb/chroma_db"
}

DEMO: QUICK VECTOR SEARCH


,rank,chunk_id,distance,source_type,file_name,source_name,preview
0,1,pdf_Farming_Schemes_0009_000009,0.387263,pdf,Farming_Schemes_pdf.txt,Farming_Schemes,support prices are a guarantee price for their...
1,2,pdf_Farming_Schemes_0010_000010,0.400016,pdf,Farming_Schemes_pdf.txt,Farming_Schemes,bumper production years. The minimum support p...
2,3,pdf_Farming_Schemes_0008_000008,0.459804,pdf,Farming_Schemes_pdf.txt,Farming_Schemes,Support Price Scheme Type of scheme : Central ...
3,4,pdf_Farming_Schemes_0192_000192,0.594758,pdf,Farming_Schemes_pdf.txt,Farming_Schemes,The maximum admissible grant for each project ...
4,5,pdf_Farming_Schemes_0331_000331,0.595863,pdf,Farming_Schemes_pdf.txt,Farming_Schemes,"Pith Unit. For a Composite or a Multiple Unit,..."


In [4]:
# ============================================================
# NOTEBOOK 3: VECTOR DB (CHROMA) + EMBEDDINGS (RAG READY)
# Input:
#  - outputs/run_<RUN_ID>/chunks/chunks.parquet
# Output:
#  - vectordb/chroma_db/   (persistent)
#  - vectordb/vector_store_report.csv
#  - vectordb/vector_store_log.json
# ============================================================

# -----------------------------
# Cell 1 — Install dependencies
# -----------------------------
!pip -q install chromadb sentence-transformers

import os, json, re
import pandas as pd
from datetime import datetime

import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

# -----------------------------
# Cell 2 — Mount Drive + paths
# -----------------------------
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

PROJECT_ROOT = "/content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project"
RUN_ID = "20260209_185402"   # ✅ same as earlier

RUN_DIR = os.path.join(PROJECT_ROOT, "outputs", f"run_{RUN_ID}")

CHUNK_DIR = os.path.join(RUN_DIR, "chunks")
CHUNKS_PARQUET = os.path.join(CHUNK_DIR, "chunks.parquet")

VECTOR_DIR = os.path.join(RUN_DIR, "vectordb")
CHROMA_DIR = os.path.join(VECTOR_DIR, "chroma_db")
os.makedirs(CHROMA_DIR, exist_ok=True)

print("RUN_DIR:", RUN_DIR)
print("CHUNKS_PARQUET:", CHUNKS_PARQUET, "| exists:", os.path.exists(CHUNKS_PARQUET))
print("CHROMA_DIR:", CHROMA_DIR)

# -----------------------------
# Cell 3 — Load chunks
# -----------------------------
if not os.path.exists(CHUNKS_PARQUET):
    raise FileNotFoundError(f"Chunks not found. Run Notebook 2 first: {CHUNKS_PARQUET}")

chunks_df = pd.read_parquet(CHUNKS_PARQUET)

print("✅ Loaded chunks:", len(chunks_df))
display(chunks_df.head(5))

# -----------------------------
# Cell 4 — Choose embedding model
# -----------------------------
# Good balanced model for English; fast and strong for RAG
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedder = SentenceTransformer(EMBED_MODEL_NAME)
print("✅ Embedding model loaded:", EMBED_MODEL_NAME)

# -----------------------------
# Cell 5 — Create / load ChromaDB
# -----------------------------
client = chromadb.PersistentClient(
    path=CHROMA_DIR,
    settings=Settings(anonymized_telemetry=False)
)

COLLECTION_NAME = f"agrigenius_{RUN_ID}"

# If rerun, delete existing collection to avoid duplicates (optional)
existing = [c.name for c in client.list_collections()]
if COLLECTION_NAME in existing:
    client.delete_collection(COLLECTION_NAME)
    print("♻️ Deleted existing collection:", COLLECTION_NAME)

collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={"project": "AgriGenius", "run_id": RUN_ID, "embedding_model": EMBED_MODEL_NAME}
)

print("✅ Collection created:", COLLECTION_NAME)

# -----------------------------
# Cell 6 — Prepare data for vector db
# -----------------------------
# Chroma accepts: ids, documents, metadatas, embeddings

docs = chunks_df["chunk_text"].astype(str).tolist()
ids = chunks_df["chunk_id"].astype(str).tolist()

metadatas = []
for _, r in chunks_df.iterrows():
    metadatas.append({
        "source_type": r["source_type"],
        "source_name": r["source_name"],
        "file_name": r["file_name"],
        "chunk_index_in_file": int(r["chunk_index_in_file"]),
        "chunk_words": int(r["chunk_words"]),
        "chunk_chars": int(r["chunk_chars"]),
    })

print("✅ Prepared documents:", len(docs))
print("Example metadata:", metadatas[0])

# -----------------------------
# Cell 7 — Embed + ingest (batched)
# -----------------------------
BATCH_SIZE = 96  # safe for Colab

def batch_iter(lst, batch_size):
    for i in range(0, len(lst), batch_size):
        yield i, lst[i:i+batch_size]

total = len(docs)
print("\n" + "="*90)
print("STEP: EMBEDDING + INDEXING INTO VECTOR DB")
print("="*90)

for start_idx, batch_docs in batch_iter(docs, BATCH_SIZE):
    batch_ids = ids[start_idx:start_idx+len(batch_docs)]
    batch_metas = metadatas[start_idx:start_idx+len(batch_docs)]

    embeddings = embedder.encode(batch_docs, show_progress_bar=False).tolist()

    collection.add(
        ids=batch_ids,
        documents=batch_docs,
        metadatas=batch_metas,
        embeddings=embeddings
    )

    print(f"Indexed {start_idx + len(batch_docs):>6}/{total}")

print("\n✅ Vector DB indexing complete.")
print("Total vectors:", collection.count())

# -----------------------------
# Cell 8 — Presentable indexing report
# -----------------------------
report_rows = []
for stype, grp in chunks_df.groupby("source_type"):
    report_rows.append({
        "source_type": stype,
        "files": grp["file_name"].nunique(),
        "chunks": len(grp),
        "avg_chunk_words": round(grp["chunk_words"].mean(), 2)
    })

report_df = pd.DataFrame(report_rows).sort_values("chunks", ascending=False)

print("\n✅ Vector DB report:")
display(report_df)

report_path = os.path.join(VECTOR_DIR, "vector_store_report.csv")
report_df.to_csv(report_path, index=False)
print("Saved:", report_path)

# -----------------------------
# Cell 9 — Save log (reproducibility)
# -----------------------------
log = {
    "run_id": RUN_ID,
    "created_at": datetime.now().isoformat(),
    "chunks_loaded": int(len(chunks_df)),
    "collection_name": COLLECTION_NAME,
    "embedding_model": EMBED_MODEL_NAME,
    "vector_db": "ChromaDB",
    "persist_path": CHROMA_DIR,
    "batch_size": BATCH_SIZE
}

log_path = os.path.join(VECTOR_DIR, "vector_store_log.json")
with open(log_path, "w", encoding="utf-8") as f:
    json.dump(log, f, indent=2)

print("✅ Log saved:", log_path)
print(json.dumps(log, indent=2))

# -----------------------------
# Cell 10 — Quick retrieval test (presentation ready)
# -----------------------------
def search(query: str, k: int = 5):
    q_emb = embedder.encode([query]).tolist()[0]
    res = collection.query(
        query_embeddings=[q_emb],
        n_results=k,
        include=["documents","metadatas","distances"]
    )

    rows = []
    for i in range(k):
        rows.append({
            "rank": i+1,
            "distance": res["distances"][0][i],
            "source_type": res["metadatas"][0][i].get("source_type"),
            "file_name": res["metadatas"][0][i].get("file_name"),
            "source_name": res["metadatas"][0][i].get("source_name"),
            "preview": res["documents"][0][i][:240] + "..."
        })
    return pd.DataFrame(rows)

print("\n" + "="*90)
print("DEMO: VECTOR SEARCH")
print("="*90)

demo_df = search("What is Minimum Support Price scheme?", k=5)
display(demo_df)


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
# ============================================================
# NOTEBOOK 3: VECTOR DB (CHROMA) — COLAB SAFE + PRESENTATION READY
# Why: Google Drive mount can cause SQLite "disk I/O error".
# Solution: Build Chroma DB in /content, then copy to Drive.
#
# Outputs (in Drive):
#  - vectordb/chroma_db/   (copied from local)
#  - vectordb/vector_store_report.csv
#  - vectordb/vector_store_log.json
# ============================================================

# -----------------------------
# Cell 1 — Install deps (safe)
# -----------------------------
!pip -q install chromadb sentence-transformers pandas pyarrow

# -----------------------------
# Cell 2 — Mount Drive + paths
# -----------------------------
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import os, json, shutil
import pandas as pd
from datetime import datetime

PROJECT_ROOT = "/content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project"
RUN_ID = "20260209_185402"  # <-- same as before (without run_)
RUN_DIR = os.path.join(PROJECT_ROOT, "outputs", f"run_{RUN_ID}")

CHUNKS_PARQUET = os.path.join(RUN_DIR, "chunks", "chunks.parquet")
VECTOR_OUT = os.path.join(RUN_DIR, "vectordb")
CHROMA_DIR_DRIVE = os.path.join(VECTOR_OUT, "chroma_db")   # final storage in Drive (copy here)

# Build locally first (SAFE)
CHROMA_DIR_LOCAL = f"/content/chroma_db_{RUN_ID}"

os.makedirs(VECTOR_OUT, exist_ok=True)

print("RUN_DIR:", RUN_DIR)
print("CHUNKS_PARQUET:", CHUNKS_PARQUET, "| exists:", os.path.exists(CHUNKS_PARQUET))
print("CHROMA_DIR_LOCAL:", CHROMA_DIR_LOCAL)
print("CHROMA_DIR_DRIVE:", CHROMA_DIR_DRIVE)

# -----------------------------
# Cell 3 — Load chunks (and verify)
# -----------------------------
if not os.path.exists(CHUNKS_PARQUET):
    raise FileNotFoundError(f"chunks.parquet not found: {CHUNKS_PARQUET}")

chunks_df = pd.read_parquet(CHUNKS_PARQUET)

# Basic sanity checks
req_cols = ["chunk_id", "chunk_text", "source_type", "source_name", "file_name", "chunk_index_in_file"]
missing = [c for c in req_cols if c not in chunks_df.columns]
if missing:
    raise ValueError(f"Missing required columns in chunks.parquet: {missing}")

chunks_df["chunk_text"] = chunks_df["chunk_text"].astype(str)
chunks_df = chunks_df[chunks_df["chunk_text"].str.strip().astype(bool)].copy()

print("✅ Loaded chunks:", len(chunks_df))
display(chunks_df.head(5))

# -----------------------------
# Cell 4 — Embedding model
# -----------------------------
from sentence_transformers import SentenceTransformer

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBED_MODEL_NAME)

print("✅ Embedding model loaded:", EMBED_MODEL_NAME)

# -----------------------------
# Cell 5 — Create fresh LOCAL Chroma DB (avoid Drive I/O)
# -----------------------------
import chromadb
from chromadb.config import Settings

# Clean local if exists (important for re-runs)
if os.path.exists(CHROMA_DIR_LOCAL):
    shutil.rmtree(CHROMA_DIR_LOCAL)

client = chromadb.PersistentClient(
    path=CHROMA_DIR_LOCAL,
    settings=Settings(anonymized_telemetry=False)
)

COLLECTION_NAME = f"agrigenius_{RUN_ID}"
collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}  # good default for MiniLM
)

print("✅ Collection ready:", COLLECTION_NAME)

# -----------------------------
# Cell 6 — Prepare docs + metadata
# -----------------------------
docs = chunks_df["chunk_text"].tolist()
ids = chunks_df["chunk_id"].astype(str).tolist()

metas = []
for _, r in chunks_df.iterrows():
    metas.append({
        "source_type": str(r["source_type"]),
        "source_name": str(r["source_name"]),
        "file_name": str(r["file_name"]),
        "chunk_index_in_file": int(r["chunk_index_in_file"]),
        "chunk_words": int(r.get("chunk_words", len(str(r["chunk_text"]).split()))),
        "chunk_chars": int(r.get("chunk_chars", len(str(r["chunk_text"])))),
    })

print("✅ Prepared documents:", len(docs))
print("Example metadata:", metas[0])

# -----------------------------
# Cell 7 — Embed + add in batches (presentation logs)
# -----------------------------
BATCH_SIZE = 96

print("\n" + "="*90)
print("STEP: EMBEDDING + INDEXING INTO VECTOR DB (LOCAL)")
print("="*90)

for start in range(0, len(docs), BATCH_SIZE):
    end = min(start + BATCH_SIZE, len(docs))
    batch_docs = docs[start:end]
    batch_ids = ids[start:end]
    batch_metas = metas[start:end]

    embs = embedder.encode(batch_docs, show_progress_bar=False).tolist()
    collection.add(documents=batch_docs, embeddings=embs, metadatas=batch_metas, ids=batch_ids)

    if (end % (BATCH_SIZE * 2) == 0) or (end == len(docs)):
        print(f"Indexed {end:>5}/{len(docs)}")

print("\n✅ Vector DB indexing complete (LOCAL).")
print("Total vectors:", collection.count())

# -----------------------------
# Cell 8 — Copy LOCAL DB to Drive (safe export step)
# -----------------------------
# If a previous DB exists in Drive, remove it (corrupted DBs will keep failing)
if os.path.exists(CHROMA_DIR_DRIVE):
    shutil.rmtree(CHROMA_DIR_DRIVE)

shutil.copytree(CHROMA_DIR_LOCAL, CHROMA_DIR_DRIVE)

print("✅ Copied Chroma DB to Drive:")
print("LOCAL:", CHROMA_DIR_LOCAL)
print("DRIVE:", CHROMA_DIR_DRIVE)

# -----------------------------
# Cell 9 — Save reports + logs (presentation-ready)
# -----------------------------
report_df = chunks_df.groupby("source_type", as_index=False).agg(
    files=("file_name", "nunique"),
    chunks=("chunk_id", "count"),
    avg_chunk_words=("chunk_words", "mean"),
)
report_path = os.path.join(VECTOR_OUT, "vector_store_report.csv")
report_df.to_csv(report_path, index=False)

log = {
    "run_id": RUN_ID,
    "created_at": datetime.now().isoformat(),
    "collection_name": COLLECTION_NAME,
    "embedding_model": EMBED_MODEL_NAME,
    "vector_db": "ChromaDB",
    "local_persist_path": CHROMA_DIR_LOCAL,
    "drive_persist_path": CHROMA_DIR_DRIVE,
    "chunks_indexed": int(len(docs)),
    "batch_size": BATCH_SIZE
}
log_path = os.path.join(VECTOR_OUT, "vector_store_log.json")
with open(log_path, "w", encoding="utf-8") as f:
    json.dump(log, f, indent=2)

print("\n✅ Vector DB report saved:", report_path)
display(report_df)

print("✅ Log saved:", log_path)
print(json.dumps(log, indent=2))

# -----------------------------
# Cell 10 — Demo search (for presentation)
# -----------------------------
def demo_search(query: str, k: int = 5):
    q_emb = embedder.encode([query]).tolist()
    res = collection.query(
        query_embeddings=q_emb,
        n_results=k,
        include=["metadatas", "documents", "distances"]
    )

    rows = []
    for i in range(len(res["ids"][0])):
        rows.append({
            "rank": i + 1,
            "distance": res["distances"][0][i],
            "source_type": res["metadatas"][0][i].get("source_type"),
            "file_name": res["metadatas"][0][i].get("file_name"),
            "source_name": res["metadatas"][0][i].get("source_name"),
            "preview": res["documents"][0][i][:140].replace("\n", " ") + "..."
        })
    return pd.DataFrame(rows)

print("\n" + "="*90)
print("DEMO: VECTOR SEARCH")
print("="*90)

display(demo_search("minimum support price scheme", k=5))


Mounted at /content/drive
RUN_DIR: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402
CHUNKS_PARQUET: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/chunks/chunks.parquet | exists: True
CHROMA_DIR_LOCAL: /content/chroma_db_20260209_185402
CHROMA_DIR_DRIVE: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/vectordb/chroma_db
✅ Loaded chunks: 891


,chunk_id,source_type,source_name,file_name,file_path,chunk_index_in_file,chunk_size_words,overlap_words,chunk_words,chunk_chars,chunk_preview,chunk_text
0,pdf_Farming_Schemes_0001_000001,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,1,220,50,220,1514,COMPENDIUM OF GOVERNMENT OF INDIA SCHEMES AND ...,COMPENDIUM OF GOVERNMENT OF INDIA SCHEMES AND ...
1,pdf_Farming_Schemes_0002_000002,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,2,220,50,220,1440,Gramin Agricultural Markets (GrAMs) 39 2. Mini...,Gramin Agricultural Markets (GrAMs) 39 2. Mini...
2,pdf_Farming_Schemes_0003_000003,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,3,220,50,220,1466,North East Special Infrastructure Development ...,North East Special Infrastructure Development ...
3,pdf_Farming_Schemes_0004_000004,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,4,220,50,220,1531,Operational Guidelines of the Scheme for Integ...,Operational Guidelines of the Scheme for Integ...
4,pdf_Farming_Schemes_0005_000005,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,5,220,50,220,1528,National TB Control Programme (RNTCP) 158 12. ...,National TB Control Programme (RNTCP) 158 12. ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model loaded: sentence-transformers/all-MiniLM-L6-v2
✅ Collection ready: agrigenius_20260209_185402
✅ Prepared documents: 891
Example metadata: {'source_type': 'pdf', 'source_name': 'Farming_Schemes', 'file_name': 'Farming_Schemes_pdf.txt', 'chunk_index_in_file': 1, 'chunk_words': 220, 'chunk_chars': 1514}

STEP: EMBEDDING + INDEXING INTO VECTOR DB (LOCAL)
Indexed   192/891
Indexed   384/891
Indexed   576/891
Indexed   768/891
Indexed   891/891

✅ Vector DB indexing complete (LOCAL).
Total vectors: 891
✅ Copied Chroma DB to Drive:
LOCAL: /content/chroma_db_20260209_185402
DRIVE: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/vectordb/chroma_db

✅ Vector DB report saved: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/vectordb/vector_store_report.csv


,source_type,files,chunks,avg_chunk_words
0,pdf,2,817,219.660955
1,web,10,74,207.270270


✅ Log saved: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/vectordb/vector_store_log.json
{
  "run_id": "20260209_185402",
  "created_at": "2026-02-10T05:59:04.016581",
  "collection_name": "agrigenius_20260209_185402",
  "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
  "vector_db": "ChromaDB",
  "local_persist_path": "/content/chroma_db_20260209_185402",
  "drive_persist_path": "/content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/vectordb/chroma_db",
  "chunks_indexed": 891,
  "batch_size": 96
}

DEMO: VECTOR SEARCH


,rank,distance,source_type,file_name,source_name,preview
0,1,0.419113,pdf,Farming_Schemes_pdf.txt,Farming_Schemes,support prices are a guarantee price for their...
1,2,0.447761,pdf,Farming_Schemes_pdf.txt,Farming_Schemes,bumper production years. The minimum support p...
2,3,0.472290,pdf,Farming_Schemes_pdf.txt,Farming_Schemes,Support Price Scheme Type of scheme : Central ...
3,4,0.620816,pdf,Farming_Schemes_pdf.txt,Farming_Schemes,All States and Union Territories For detail In...
4,5,0.631299,pdf,Farming_Schemes_pdf.txt,Farming_Schemes,The maximum admissible grant for each project ...


In [ ]:
# ============================================================
# NOTEBOOK 4: VECTOR DB + SEARCH + (OPTIONAL) RAG DEMO (FINAL)
# Works with your outputs from:
#  - Notebook 2 Chunking -> chunks/chunks.parquet
#  - Notebook 3 VectorDB -> vectordb/chroma_db (optional to rebuild)
#
# Fix included:
#  ✅ Chroma query() include does NOT accept "ids" in some versions
#     (ids are always returned in res["ids"])
#
# Outputs (saved in Drive):
#  - vectordb/vector_store_report.csv
#  - vectordb/vector_store_log.json
#  - rag_demo/search_results.csv
#  - rag_demo/search_results.json
#  - rag_demo/query_log.json
# ============================================================

# -----------------------------
# Cell 1 — Install deps (safe)
# -----------------------------
!pip -q install chromadb sentence-transformers pandas pyarrow

# -----------------------------
# Cell 2 — Mount Drive (safe)
# -----------------------------
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import os, json, shutil, time
import pandas as pd
from datetime import datetime

# -----------------------------
# Cell 3 — Set PROJECT + RUN
# -----------------------------
PROJECT_ROOT = "/content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project"

# ✅ MUST match your run id (WITHOUT "run_")
RUN_ID = "20260209_185402"

RUN_DIR = os.path.join(PROJECT_ROOT, "outputs", f"run_{RUN_ID}")

CHUNK_OUT = os.path.join(RUN_DIR, "chunks")
CHUNKS_PARQUET = os.path.join(CHUNK_OUT, "chunks.parquet")

VDB_OUT = os.path.join(RUN_DIR, "vectordb")
os.makedirs(VDB_OUT, exist_ok=True)

# For safe persistence (Drive can cause I/O errors), we index locally then copy to Drive
CHROMA_DIR_LOCAL = f"/content/chroma_db_{RUN_ID}"
CHROMA_DIR_DRIVE = os.path.join(VDB_OUT, "chroma_db")

RAG_OUT = os.path.join(RUN_DIR, "rag_demo")
os.makedirs(RAG_OUT, exist_ok=True)

print("Mounted at /content/drive")
print("RUN_DIR:", RUN_DIR)
print("CHUNKS_PARQUET:", CHUNKS_PARQUET, "| exists:", os.path.exists(CHUNKS_PARQUET))
print("CHROMA_DIR_LOCAL:", CHROMA_DIR_LOCAL)
print("CHROMA_DIR_DRIVE:", CHROMA_DIR_DRIVE)
print("RAG_OUT:", RAG_OUT)


# -----------------------------
# Cell 4 — Load chunks
# -----------------------------
if not os.path.exists(CHUNKS_PARQUET):
    raise FileNotFoundError(f"Missing chunks.parquet at: {CHUNKS_PARQUET}")

chunks_df = pd.read_parquet(CHUNKS_PARQUET)

# Safety: ensure required cols exist
required_cols = ["chunk_id","chunk_text","source_type","source_name","file_name","chunk_index_in_file","chunk_words","chunk_chars"]
missing = [c for c in required_cols if c not in chunks_df.columns]
if missing:
    raise ValueError(f"chunks.parquet missing columns: {missing}")

# Drop blank rows if any
chunks_df["chunk_text"] = chunks_df["chunk_text"].fillna("").astype(str)
chunks_df = chunks_df[chunks_df["chunk_text"].str.strip() != ""].copy()

print("✅ Loaded chunks:", len(chunks_df))
display(chunks_df.head(5))


# -----------------------------
# Cell 5 — Embedding model
# -----------------------------
from sentence_transformers import SentenceTransformer

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBED_MODEL)

print("✅ Embedding model loaded:", EMBED_MODEL)


# -----------------------------
# Cell 6 — ChromaDB setup
# -----------------------------
import chromadb
from chromadb.config import Settings

# Clean local folder if you want to rebuild every time:
REBUILD_DB = True  # set False to reuse existing local db if present

if REBUILD_DB and os.path.exists(CHROMA_DIR_LOCAL):
    shutil.rmtree(CHROMA_DIR_LOCAL)

os.makedirs(CHROMA_DIR_LOCAL, exist_ok=True)

client = chromadb.PersistentClient(
    path=CHROMA_DIR_LOCAL,
    settings=Settings(anonymized_telemetry=False)
)

COLLECTION_NAME = f"agrigenius_{RUN_ID}"

# Recreate collection cleanly if rebuild
if REBUILD_DB:
    try:
        client.delete_collection(COLLECTION_NAME)
    except:
        pass

collection = client.get_or_create_collection(name=COLLECTION_NAME)
print("✅ Collection ready:", COLLECTION_NAME)


# -----------------------------
# Cell 7 — Prepare docs + metadata
# -----------------------------
docs = chunks_df["chunk_text"].tolist()
ids = chunks_df["chunk_id"].tolist()

metas = []
for _, r in chunks_df.iterrows():
    metas.append({
        "source_type": r["source_type"],
        "source_name": r["source_name"],
        "file_name": r["file_name"],
        "chunk_index_in_file": int(r["chunk_index_in_file"]),
        "chunk_words": int(r["chunk_words"]),
        "chunk_chars": int(r["chunk_chars"]),
    })

print("✅ Prepared documents:", len(docs))
print("Example metadata:", metas[0])


# -----------------------------
# Cell 8 — Index into Chroma (with batching)
# -----------------------------
BATCH_SIZE = 96
print("\n" + "="*90)
print("STEP: EMBEDDING + INDEXING INTO VECTOR DB (LOCAL)")
print("="*90)

# Optional: avoid duplicate adds if collection already has vectors
existing = 0
try:
    existing = collection.count()
except:
    existing = 0

if existing > 0 and not REBUILD_DB:
    print(f"⚠️ Collection already has {existing} vectors. Skipping re-index.")
else:
    for start in range(0, len(docs), BATCH_SIZE):
        end = min(start + BATCH_SIZE, len(docs))

        batch_docs = docs[start:end]
        batch_ids = ids[start:end]
        batch_metas = metas[start:end]

        # Embed
        batch_emb = embedder.encode(batch_docs, show_progress_bar=False).tolist()

        # Add
        collection.add(
            ids=batch_ids,
            embeddings=batch_emb,
            documents=batch_docs,
            metadatas=batch_metas
        )

        if (end % (BATCH_SIZE * 2) == 0) or (end == len(docs)):
            print(f"Indexed {end:>5}/{len(docs)}")

    print("\n✅ Vector DB indexing complete (LOCAL).")
    print("Total vectors:", collection.count())


# -----------------------------
# Cell 9 — Copy DB to Drive (safe)
# -----------------------------
# Drive can throw disk I/O errors if used directly for Chroma.
# So we persist locally and then copy final folder to Drive.
if os.path.exists(CHROMA_DIR_DRIVE):
    shutil.rmtree(CHROMA_DIR_DRIVE)

shutil.copytree(CHROMA_DIR_LOCAL, CHROMA_DIR_DRIVE)

print("✅ Copied Chroma DB to Drive:")
print("LOCAL:", CHROMA_DIR_LOCAL)
print("DRIVE:", CHROMA_DIR_DRIVE)


# -----------------------------
# Cell 10 — Vector DB report + logs
# -----------------------------
report_df = chunks_df.groupby("source_type", as_index=False).agg(
    files=("file_name","nunique"),
    chunks=("chunk_id","count"),
    avg_chunk_words=("chunk_words","mean")
)

report_path = os.path.join(VDB_OUT, "vector_store_report.csv")
report_df.to_csv(report_path, index=False)

log = {
    "run_id": RUN_ID,
    "created_at": datetime.now().isoformat(),
    "collection_name": COLLECTION_NAME,
    "embedding_model": EMBED_MODEL,
    "vector_db": "ChromaDB",
    "local_persist_path": CHROMA_DIR_LOCAL,
    "drive_persist_path": CHROMA_DIR_DRIVE,
    "chunks_indexed": int(len(chunks_df)),
    "batch_size": BATCH_SIZE
}
log_path = os.path.join(VDB_OUT, "vector_store_log.json")
with open(log_path, "w", encoding="utf-8") as f:
    json.dump(log, f, indent=2)

print("\n✅ Vector DB report saved:", report_path)
display(report_df)
print("\n✅ Log saved:", log_path)
print(json.dumps(log, indent=2))


# -----------------------------
# Cell 11 — Search function (FIXED include list)
# -----------------------------
def search_vector_db(query: str, k: int = 5):
    q_emb = embedder.encode([query]).tolist()

    # ✅ FIX: do NOT include "ids" in include list
    res = collection.query(
        query_embeddings=q_emb,
        n_results=k,
        include=["metadatas", "documents", "distances"]
    )

    rows = []
    for i in range(len(res["ids"][0])):  # ids always exist here
        meta = res["metadatas"][0][i] if res.get("metadatas") else {}
        doc = res["documents"][0][i] if res.get("documents") else ""
        dist = res["distances"][0][i] if res.get("distances") else None

        rows.append({
            "rank": i + 1,
            "chunk_id": res["ids"][0][i],
            "distance": float(dist) if dist is not None else None,
            "source_type": meta.get("source_type"),
            "file_name": meta.get("file_name"),
            "source_name": meta.get("source_name"),
            "chunk_index_in_file": meta.get("chunk_index_in_file"),
            "preview": (doc[:240].replace("\n"," ") + "...") if doc else ""
        })

    return pd.DataFrame(rows), res


# -----------------------------
# Cell 12 — DEMO SEARCH (presentation-ready)
# -----------------------------
QUERY = "What is Minimum Support Price scheme and how does it work?"
TOP_K = 5

results_df, raw_res = search_vector_db(QUERY, k=TOP_K)

print("\n" + "="*90)
print("DEMO: VECTOR SEARCH")
print("="*90)
display(results_df)

# Save search outputs
results_csv = os.path.join(RAG_OUT, "search_results.csv")
results_json = os.path.join(RAG_OUT, "search_results.json")

results_df.to_csv(results_csv, index=False)
with open(results_json, "w", encoding="utf-8") as f:
    json.dump(raw_res, f, indent=2)

query_log = {
    "run_id": RUN_ID,
    "created_at": datetime.now().isoformat(),
    "query": QUERY,
    "top_k": TOP_K,
    "saved_csv": results_csv,
    "saved_json": results_json
}
query_log_path = os.path.join(RAG_OUT, "query_log.json")
with open(query_log_path, "w", encoding="utf-8") as f:
    json.dump(query_log, f, indent=2)

print("\n✅ Saved:", results_csv)
print("✅ Saved:", results_json)
print("✅ Saved:", query_log_path)


# -----------------------------
# Cell 13 — OPTIONAL: Simple "answer" using top chunks (no LLM)
# -----------------------------
def build_cited_answer(query: str, results_df: pd.DataFrame, raw_res: dict, max_sources: int = 3):
    """
    No external LLM used (presentation-safe).
    Creates a short answer draft + citations for guide.
    """
    answer_lines = []
    answer_lines.append(f"Query: {query}\n")

    picked = results_df.head(max_sources)
    answer_lines.append("Evidence snippets:\n")

    # pull documents from raw_res
    docs = raw_res.get("documents", [[]])[0]
    metas = raw_res.get("metadatas", [[]])[0]
    ids = raw_res.get("ids", [[]])[0]

    for i in range(min(max_sources, len(ids))):
        chunk_id = ids[i]
        meta = metas[i] if i < len(metas) else {}
        doc = docs[i] if i < len(docs) else ""

        cite = f"[{i+1}] {meta.get('file_name')} | {meta.get('source_name')} | chunk {meta.get('chunk_index_in_file')} | id={chunk_id}"
        snippet = doc[:500].replace("\n"," ").strip()
        answer_lines.append(cite)
        answer_lines.append(snippet)
        answer_lines.append("")

    return "\n".join(answer_lines).strip()

draft_answer = build_cited_answer(QUERY, results_df, raw_res, max_sources=3)

print("\n" + "="*90)
print("OPTIONAL: CITED ANSWER DRAFT (NO LLM)")
print("="*90)
print(draft_answer)

answer_path = os.path.join(RAG_OUT, "cited_answer_draft.txt")
with open(answer_path, "w", encoding="utf-8") as f:
    f.write(draft_answer)

print("\n✅ Saved:", answer_path)


Mounted at /content/drive
Mounted at /content/drive
RUN_DIR: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402
CHUNKS_PARQUET: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/chunks/chunks.parquet | exists: True
CHROMA_DIR_LOCAL: /content/chroma_db_20260209_185402
CHROMA_DIR_DRIVE: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/vectordb/chroma_db
RAG_OUT: /content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project/outputs/run_20260209_185402/rag_demo
✅ Loaded chunks: 891


,chunk_id,source_type,source_name,file_name,file_path,chunk_index_in_file,chunk_size_words,overlap_words,chunk_words,chunk_chars,chunk_preview,chunk_text
0,pdf_Farming_Schemes_0001_000001,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,1,220,50,220,1514,COMPENDIUM OF GOVERNMENT OF INDIA SCHEMES AND ...,COMPENDIUM OF GOVERNMENT OF INDIA SCHEMES AND ...
1,pdf_Farming_Schemes_0002_000002,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,2,220,50,220,1440,Gramin Agricultural Markets (GrAMs) 39 2. Mini...,Gramin Agricultural Markets (GrAMs) 39 2. Mini...
2,pdf_Farming_Schemes_0003_000003,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,3,220,50,220,1466,North East Special Infrastructure Development ...,North East Special Infrastructure Development ...
3,pdf_Farming_Schemes_0004_000004,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,4,220,50,220,1531,Operational Guidelines of the Scheme for Integ...,Operational Guidelines of the Scheme for Integ...
4,pdf_Farming_Schemes_0005_000005,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,5,220,50,220,1528,National TB Control Programme (RNTCP) 158 12. ...,National TB Control Programme (RNTCP) 158 12. ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model loaded: sentence-transformers/all-MiniLM-L6-v2
✅ Collection ready: agrigenius_20260209_185402
✅ Prepared documents: 891
Example metadata: {'source_type': 'pdf', 'source_name': 'Farming_Schemes', 'file_name': 'Farming_Schemes_pdf.txt', 'chunk_index_in_file': 1, 'chunk_words': 220, 'chunk_chars': 1514}

STEP: EMBEDDING + INDEXING INTO VECTOR DB (LOCAL)


InternalError: Query error: Database error: error returned from database: (code: 1032) attempt to write a readonly database

In [ ]:
# ============================================================
# NOTEBOOK 3: VECTOR DB BUILD (FINAL STABLE + PRESENTATION READY)
# Output:
#  - vectordb/chroma_db (copied to Drive at end)
#  - vectordb/vector_store_report.csv
#  - vectordb/vector_store_log.json
# ============================================================

# -----------------------------
# Cell 1 — Install deps (safe)
# -----------------------------
!pip -q install chromadb==0.5.5 sentence-transformers==3.0.1 pandas pyarrow

# -----------------------------
# Cell 2 — Mount Drive + paths
# -----------------------------
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import os, json, shutil
import pandas as pd
from datetime import datetime

PROJECT_ROOT = "/content/drive/MyDrive/Colab Notebooks/Shruti/AgriGenius_Project"
RUN_ID = "20260209_185402"  # <-- same RUN_ID you used earlier (WITHOUT "run_")

RUN_DIR = os.path.join(PROJECT_ROOT, "outputs", f"run_{RUN_ID}")
CHUNKS_PARQUET = os.path.join(RUN_DIR, "chunks", "chunks.parquet")

VDB_OUT = os.path.join(RUN_DIR, "vectordb")
os.makedirs(VDB_OUT, exist_ok=True)

# Local persistent DB path (fast + safe)
CHROMA_DIR_LOCAL = f"/content/chroma_db_{RUN_ID}"

# Drive copy target (for storage)
CHROMA_DIR_DRIVE = os.path.join(VDB_OUT, "chroma_db")

print("RUN_DIR:", RUN_DIR)
print("CHUNKS_PARQUET:", CHUNKS_PARQUET, "| exists:", os.path.exists(CHUNKS_PARQUET))
print("CHROMA_DIR_LOCAL:", CHROMA_DIR_LOCAL)
print("CHROMA_DIR_DRIVE:", CHROMA_DIR_DRIVE)

# -----------------------------
# Cell 3 — Load chunks
# -----------------------------
chunks_df = pd.read_parquet(CHUNKS_PARQUET)
print("✅ Loaded chunks:", len(chunks_df))
display(chunks_df.head(5))

# -----------------------------
# Cell 4 — Embedding model
# -----------------------------
from sentence_transformers import SentenceTransformer

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBED_MODEL_NAME)
print("✅ Embedding model loaded:", EMBED_MODEL_NAME)

# -----------------------------
# Cell 5 — Create fresh local Chroma DB
# -----------------------------
import chromadb
from chromadb.config import Settings

# Clean local dir if exists (prevents readonly / corruption conflicts)
if os.path.exists(CHROMA_DIR_LOCAL):
    shutil.rmtree(CHROMA_DIR_LOCAL)

client = chromadb.PersistentClient(
    path=CHROMA_DIR_LOCAL,
    settings=Settings(anonymized_telemetry=False)
)

COLLECTION_NAME = f"agrigenius_{RUN_ID}"
collection = client.get_or_create_collection(name=COLLECTION_NAME)
print("✅ Collection ready:", COLLECTION_NAME)

# -----------------------------
# Cell 6 — Prepare docs + metadata
# -----------------------------
docs = chunks_df["chunk_text"].astype(str).tolist()
ids  = chunks_df["chunk_id"].astype(str).tolist()

metadatas = []
for _, r in chunks_df.iterrows():
    metadatas.append({
        "source_type": str(r["source_type"]),
        "source_name": str(r["source_name"]),
        "file_name": str(r["file_name"]),
        "chunk_index_in_file": int(r["chunk_index_in_file"]),
        "chunk_words": int(r["chunk_words"]),
        "chunk_chars": int(r["chunk_chars"]),
    })

print("✅ Prepared documents:", len(docs))
print("Example metadata:", metadatas[0])

# -----------------------------
# Cell 7 — Index in batches (presentation friendly)
# -----------------------------
BATCH_SIZE = 96

print("\n" + "="*90)
print("STEP: EMBEDDING + INDEXING INTO VECTOR DB (LOCAL)")
print("="*90)

for start in range(0, len(docs), BATCH_SIZE):
    end = min(start + BATCH_SIZE, len(docs))
    batch_docs = docs[start:end]
    batch_ids  = ids[start:end]
    batch_meta = metadatas[start:end]

    batch_emb = embedder.encode(batch_docs, show_progress_bar=False).tolist()

    collection.add(
        ids=batch_ids,
        documents=batch_docs,
        metadatas=batch_meta,
        embeddings=batch_emb
    )

    print(f"Indexed {end:>5}/{len(docs)}")

print("\n✅ Vector DB indexing complete (LOCAL).")
print("Total vectors:", collection.count())

# -----------------------------
# Cell 8 — Copy DB to Drive safely
# -----------------------------
if os.path.exists(CHROMA_DIR_DRIVE):
    shutil.rmtree(CHROMA_DIR_DRIVE)

shutil.copytree(CHROMA_DIR_LOCAL, CHROMA_DIR_DRIVE)

print("✅ Copied Chroma DB to Drive:")
print("LOCAL:", CHROMA_DIR_LOCAL)
print("DRIVE:", CHROMA_DIR_DRIVE)

# -----------------------------
# Cell 9 — Reports + log
# -----------------------------
report_df = chunks_df.groupby("source_type", as_index=False).agg(
    files=("file_name", "nunique"),
    chunks=("chunk_id", "count"),
    avg_chunk_words=("chunk_words", "mean")
)

report_path = os.path.join(VDB_OUT, "vector_store_report.csv")
report_df.to_csv(report_path, index=False)
print("✅ Vector DB report saved:", report_path)
display(report_df)

log = {
    "run_id": RUN_ID,
    "created_at": datetime.now().isoformat(),
    "collection_name": COLLECTION_NAME,
    "embedding_model": EMBED_MODEL_NAME,
    "vector_db": "ChromaDB",
    "local_persist_path": CHROMA_DIR_LOCAL,
    "drive_persist_path": CHROMA_DIR_DRIVE,
    "chunks_indexed": int(len(docs)),
    "batch_size": BATCH_SIZE
}

log_path = os.path.join(VDB_OUT, "vector_store_log.json")
with open(log_path, "w", encoding="utf-8") as f:
    json.dump(log, f, indent=2)

print("✅ Log saved:", log_path)
print(json.dumps(log, indent=2))


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 584.3/584.3 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.1/227.1 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 5.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.37.0 requires n

,chunk_id,source_type,source_name,file_name,file_path,chunk_index_in_file,chunk_size_words,overlap_words,chunk_words,chunk_chars,chunk_preview,chunk_text
0,pdf_Farming_Schemes_0001_000001,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,1,220,50,220,1514,COMPENDIUM OF GOVERNMENT OF INDIA SCHEMES AND ...,COMPENDIUM OF GOVERNMENT OF INDIA SCHEMES AND ...
1,pdf_Farming_Schemes_0002_000002,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,2,220,50,220,1440,Gramin Agricultural Markets (GrAMs) 39 2. Mini...,Gramin Agricultural Markets (GrAMs) 39 2. Mini...
2,pdf_Farming_Schemes_0003_000003,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,3,220,50,220,1466,North East Special Infrastructure Development ...,North East Special Infrastructure Development ...
3,pdf_Farming_Schemes_0004_000004,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,4,220,50,220,1531,Operational Guidelines of the Scheme for Integ...,Operational Guidelines of the Scheme for Integ...
4,pdf_Farming_Schemes_0005_000005,pdf,Farming_Schemes,Farming_Schemes_pdf.txt,/content/drive/MyDrive/Colab Notebooks/Shruti/...,5,220,50,220,1528,National TB Control Programme (RNTCP) 158 12. ...,National TB Control Programme (RNTCP) 158 12. ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model loaded: sentence-transformers/all-MiniLM-L6-v2


InternalError: Database error: error returned from database: (code: 14) unable to open database file